# Transport-Only Simulation with Synthetic Forcing

This notebook simulates pure transport (advection + diffusion) without any biological reactions.

**Configuration:**
- Domain: 360×139 at 1° resolution (synthetic forcing)
- Period: 5 years (2000-2005), daily timestep
- Initial state: Gaussian biomass blob from `biomass_initial.nc`
- Forcing: Synthetic vortex from `06_a_Synthetic_LMTL.ipynb`
- Transport: Advection + Diffusion (D=0 by default, configurable)
- Boundaries: CLOSED (configurable to PERIODIC)

In [ ]:
from datetime import timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pint
import xarray as xr
from IPython.display import Markdown

from seapopym.blueprint import Blueprint
from seapopym.controller import SimulationConfig, SimulationController
from seapopym.standard.coordinates import Coordinates
from seapopym.transport import (
    BoundaryConditions,
    BoundaryType,
    check_diffusion_stability,
    compute_advection_cfl,
    compute_spherical_cell_areas,
    compute_spherical_dx,
    compute_spherical_dy,
    compute_spherical_face_areas_ew,
    compute_spherical_face_areas_ns,
    compute_transport_numba,
)

ureg = pint.get_application_registry()
print("✅ Imports successful")

## 1. Configuration

In [ ]:
# Paths to synthetic forcing data
data_folder = Path("./LMTL_Synthetic")
forcing_folder = data_folder / "data"
initial_conditions_folder = data_folder / "initial_conditions"
mask_file = data_folder / "mask.nc"
biomass_initial_file = initial_conditions_folder / "ZPK_D1N1_biomass_19991231.nc"

# Simulation configuration
start_date = "2000-01-01"
end_date = "2001-01-01"  # Match the synthetic data range (not 2005-01-01)
timestep = timedelta(hours=3)  # Daily timestep

config = SimulationConfig(
    start_date=start_date,
    end_date=end_date,
    timestep=timestep,
)

# Transport parameters (configurable)
D_horizontal = ureg.Quantity(500, ureg.m**2 / ureg.s)  # No diffusion by default

print(f"Simulation period: {start_date} to {end_date}")
print(f"Timestep: {timestep}")
print(f"Diffusion coefficient: D = {D_horizontal}")

## 2. Load Synthetic Forcing Data

In [ ]:
# Load mask
mask_ds = xr.open_dataset(mask_file)
ocean_mask = mask_ds["mask"]

# Rename coordinates to match our convention
ocean_mask = ocean_mask.rename({"latitude": Coordinates.Y.value, "longitude": Coordinates.X.value})

print(f"Mask shape: {ocean_mask.shape}")
print(f"Ocean cells: {(ocean_mask == 1).sum().values}")
print(f"Land cells: {(ocean_mask == 0).sum().values}")

# Visualize mask
plt.figure(figsize=(12, 6))
ocean_mask.plot(cmap="Blues")
plt.title("Ocean/Land Mask (1=ocean, 0=land)")
plt.show()

In [ ]:
# Load one forcing file to get the structure and coordinates
sample_file = forcing_folder / "data_20000101.nc"
sample_ds = xr.open_dataset(sample_file)

print("Sample forcing file structure:")
print(sample_ds)

# Extract coordinates
lats = sample_ds["latitude"].values
lons = sample_ds["longitude"].values
depth = sample_ds["depth"].values

print(f"\nDomain: {len(lons)}°lon × {len(lats)}°lat × {len(depth)} depth")

In [ ]:
# Load all forcing files and concatenate along time
import pandas as pd

# Generate list of dates (include end_date since we want the full range)
time_range = pd.date_range(start_date, end_date, freq="D")

print(f"Loading {len(time_range)} forcing files...")

# Load currents from all files
forcing_files = [forcing_folder / f"data_{date.strftime('%Y%m%d')}.nc" for date in time_range]

# Open all files and concatenate
forcings = xr.open_mfdataset(
    forcing_files,
    concat_dim="time",
    combine="nested",
    coords="minimal",
    compat="override",
)

# Rename coordinates
forcings = forcings.rename(
    {
        "latitude": Coordinates.Y.value,
        "longitude": Coordinates.X.value,
        "depth": Coordinates.Z.value,
    }
)

# Add proper time coordinate
forcings = forcings.assign_coords({Coordinates.T.value: time_range})

# Select only currents (we don't need temperature and primary_production for pure transport)
forcings = forcings[["current_u", "current_v"]]

# IMPORTANT: Load data into memory to avoid Dask chunking issues
print("Loading data into memory...")
forcings = forcings.load()

print(f"\nForcing data loaded:")
print(forcings)

In [ ]:
# Visualize velocity field at t=0
u_t0 = forcings["current_u"].isel({Coordinates.T.value: 0, Coordinates.Z.value: 0})
v_t0 = forcings["current_v"].isel({Coordinates.T.value: 0, Coordinates.Z.value: 0})
speed_t0 = np.sqrt(u_t0**2 + v_t0**2)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# U component
u_t0.plot(ax=axes[0], cmap="RdBu_r")
axes[0].set_title("Zonal velocity (u)")

# V component
v_t0.plot(ax=axes[1], cmap="RdBu_r")
axes[1].set_title("Meridional velocity (v)")

# Speed
speed_t0.plot(ax=axes[2], cmap="viridis")
axes[2].set_title("Speed (magnitude)")

plt.tight_layout()
plt.show()

print(f"Velocity range: u=[{u_t0.min().values:.3f}, {u_t0.max().values:.3f}] m/s")
print(f"Velocity range: v=[{v_t0.min().values:.3f}, {v_t0.max().values:.3f}] m/s")
print(f"Max speed: {speed_t0.max().values:.3f} m/s")

## 3. Load Initial Biomass State

In [ ]:
# Load initial biomass
biomass_ds = xr.open_dataset(biomass_initial_file)
biomass_init = biomass_ds["biomass"].isel(time=0)

# Rename coordinates
biomass_init = biomass_init.rename(
    {"latitude": Coordinates.Y.value, "longitude": Coordinates.X.value}
)

print(f"Initial biomass shape: {biomass_init.shape}")
print(f"Biomass range: [{biomass_init.min().values:.3e}, {biomass_init.max().values:.3e}]")
print(f"Total biomass: {biomass_init.sum().values:.3e}")

# Visualize initial state
fig, ax = plt.subplots(figsize=(12, 6))
biomass_init.plot(ax=ax, cmap="YlOrRd")
ocean_mask.plot.contour(ax=ax, levels=[0.5], colors="black", linewidths=2)
ax.set_title("Initial Biomass State")
plt.show()

## 4. Compute Grid Geometry

In [ ]:
# Get coordinate arrays
lat = forcings[Coordinates.Y.value]
lon = forcings[Coordinates.X.value]

# Compute grid metrics
cell_areas = compute_spherical_cell_areas(lat, lon)
face_areas_ew = compute_spherical_face_areas_ew(lat, lon)
face_areas_ns = compute_spherical_face_areas_ns(lat, lon)
dx = compute_spherical_dx(lat, lon)
dy = compute_spherical_dy(lat, lon)

print("Grid geometry computed:")
print(f"  Cell areas: {cell_areas.shape}")
print(f"  dx: min={dx.values.min():.0f}m, max={dx.values.max():.0f}m")
print(f"  dy: min={dy.values.min():.0f}m, max={dy.values.max():.0f}m")

## 5. Prepare Forcing Dataset for Simulation

In [ ]:
# Add grid geometry to forcings
forcings["cell_areas"] = cell_areas
forcings["face_areas_ew"] = face_areas_ew
forcings["face_areas_ns"] = face_areas_ns
forcings["dx"] = dx
forcings["dy"] = dy
forcings["ocean_mask"] = ocean_mask

# Add timestep
forcings["dt"] = config.timestep.total_seconds()

# Add boundary conditions
forcings["boundary_north"] = BoundaryType.CLOSED
forcings["boundary_south"] = BoundaryType.CLOSED
forcings["boundary_east"] = BoundaryType.PERIODIC
forcings["boundary_west"] = BoundaryType.PERIODIC

# Extract currents at single depth (since we only have 1 depth level)
forcings["current_u"] = forcings["current_u"].isel({Coordinates.Z.value: 0})
forcings["current_v"] = forcings["current_v"].isel({Coordinates.Z.value: 0})

print("Forcing dataset prepared:")
print(forcings)

## 6. Check Stability Conditions

In [ ]:
print("=" * 70)
print("STABILITY VERIFICATION")
print("=" * 70)

# Check advection CFL
stability_adv = compute_advection_cfl(
    u=forcings["current_u"].isel({Coordinates.T.value: 0}),
    v=forcings["current_v"].isel({Coordinates.T.value: 0}),
    dx=dx,
    dy=dy,
    dt=config.timestep.total_seconds(),
)

print(f"\nAdvection CFL:")
print(f"  Max CFL: {stability_adv['cfl_max']:.4f} (limit: 1.0)")
print(f"  Stable: {stability_adv['is_stable']}")

if D_horizontal.magnitude > 0:
    # Check diffusion CFL
    stability_diff = check_diffusion_stability(
        D=D_horizontal.magnitude, dx=dx, dy=dy, dt=config.timestep.total_seconds()
    )

    print(f"\nDiffusion CFL:")
    print(f"  Max CFL: {stability_diff['cfl_diffusion']:.4f} (limit: 0.25)")
    print(f"  Stable: {stability_diff['is_stable']}")
    print(f"  Safety margin: {stability_diff['margin']:.2f}x")
else:
    print(f"\nDiffusion: DISABLED (D=0)")

print("\n" + "=" * 70)

if not stability_adv["is_stable"]:
    print("⚠️  WARNING: Advection CFL condition violated!")
elif D_horizontal.magnitude > 0 and not stability_diff["is_stable"]:
    print("⚠️  WARNING: Diffusion CFL condition violated!")
else:
    print("✅ All stability conditions satisfied")

## 7. Configure Transport Model

In [ ]:
def configure_transport_model(bp):
    """Configure blueprint with transport-only dynamics."""

    # Register forcings
    bp.register_forcing(
        "current_u",
        dims=(Coordinates.T.value, Coordinates.Y.value, Coordinates.X.value),
        units="m/s",
    )
    bp.register_forcing(
        "current_v",
        dims=(Coordinates.T.value, Coordinates.Y.value, Coordinates.X.value),
        units="m/s",
    )
    bp.register_forcing(
        "ocean_mask", dims=(Coordinates.Y.value, Coordinates.X.value), units="dimensionless"
    )
    bp.register_forcing("cell_areas", dims=(Coordinates.Y.value, Coordinates.X.value), units="m**2")

    # Face areas have special dimensions (y, x_left) and (y_left, x)
    # Don't specify dims to let Blueprint accept them as-is
    bp.register_forcing("face_areas_ew", units="m")
    bp.register_forcing("face_areas_ns", units="m")

    bp.register_forcing("dx", dims=(Coordinates.Y.value, Coordinates.X.value), units="m")
    bp.register_forcing("dy", dims=(Coordinates.Y.value, Coordinates.X.value), units="m")
    bp.register_forcing("boundary_north", units="dimensionless")
    bp.register_forcing("boundary_south", units="dimensionless")
    bp.register_forcing("boundary_east", units="dimensionless")
    bp.register_forcing("boundary_west", units="dimensionless")
    bp.register_forcing("dt")

    # Register transport group
    bp.register_group(
        group_prefix="Transport",
        units=[
            {
                "func": compute_transport_numba,
                "input_mapping": {
                    "state": "biomass",
                    "u": "current_u",
                    "v": "current_v",
                    "D": "D_horizontal",
                    "dx": "dx",
                    "dy": "dy",
                    "cell_areas": "cell_areas",
                    "face_areas_ew": "face_areas_ew",
                    "face_areas_ns": "face_areas_ns",
                    "mask": "ocean_mask",
                    "boundary_north": "boundary_north",
                    "boundary_south": "boundary_south",
                    "boundary_east": "boundary_east",
                    "boundary_west": "boundary_west",
                },
                "output_mapping": {
                    "advection_rate": "biomass_advection_tendency",
                    "diffusion_rate": "biomass_diffusion_tendency",
                },
                "output_tendencies": {
                    "advection_rate": "biomass",
                    "diffusion_rate": "biomass",
                },
                "output_units": {
                    "advection_rate": "g/m**2/second",
                    "diffusion_rate": "g/m**2/second",
                },
            },
        ],
        parameters={
            "D_horizontal": {"units": "m**2/second"},
        },
        state_variables={
            "biomass": {
                "dims": (Coordinates.Y.value, Coordinates.X.value),
                "units": "g/m**2",
            },
        },
    )


bp = Blueprint()
configure_transport_model(bp)
Markdown("```mermaid\\n" + bp.export_mermaid() + "\\n```")

## 8. Prepare Initial State

In [ ]:
# Create initial state dataset
initial_state = xr.Dataset({"biomass": biomass_init})

# Ensure proper attributes (units required by Blueprint)
initial_state["biomass"].attrs = {"units": "g/m**2"}

print("Initial state:")
print(initial_state)

## 9. Run Simulation

In [ ]:
# Setup controller
controller = SimulationController(config, backend="sequential")

controller.setup(
    configure_transport_model,
    forcings,
    initial_state={"Transport": initial_state},
    parameters={"Transport": {"D_horizontal": D_horizontal}},
    output_variables={"Transport": ["biomass"]},
)

controller.run()

## 10. Analyze Results

In [ ]:
# Extract results
results = controller.results
biomass = results["Transport/biomass"]

# Compute total mass over time
total_mass = (biomass * cell_areas).sum(dim=[Coordinates.Y.value, Coordinates.X.value])

# Mass conservation
initial_mass = total_mass.isel(time=0).values
final_mass = total_mass.isel(time=-1).values
mass_error = (final_mass - initial_mass) / initial_mass * 100

print("Mass Conservation Analysis:")
print(f"  Initial mass: {initial_mass:.6e}")
print(f"  Final mass: {final_mass:.6e}")
print(f"  Relative error: {mass_error:.6e}%")

# Plot mass evolution
fig, ax = plt.subplots(figsize=(12, 5))
total_mass.plot(ax=ax)
ax.axhline(initial_mass, color="red", linestyle="--", alpha=0.5, label="Initial mass")
ax.set_title("Total Mass Evolution")
ax.set_ylabel("Total Mass")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 11. Visualize Evolution

In [ ]:
# Select snapshots to visualize (every 6 months ~ 180 days)
snapshot_indices = np.arange(0, len(biomass.time), 180 * 9)
snapshot_indices = snapshot_indices[:8]  # Max 8 snapshots

num_snapshots = len(snapshot_indices)
ncols = 4
nrows = (num_snapshots + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(20, 5 * nrows))
axes = axes.flatten()

# Find global vmax for consistent color scale
vmax = biomass.isel(time=0).max().values / 100

for i, idx in enumerate(snapshot_indices):
    ax = axes[i]
    time_val = biomass.time.isel(time=idx).values

    # Plot biomass
    biomass.isel(time=idx).plot(ax=ax, cmap="YlOrRd", vmin=0, vmax=vmax, add_colorbar=True)

    # Overlay mask
    ocean_mask.plot.contour(ax=ax, levels=[0.5], colors="black", linewidths=1, alpha=0.5)

    ax.set_title(f"t = {pd.Timestamp(time_val).strftime('%Y-%m-%d')}")
    ax.set_xlabel("")
    ax.set_ylabel("")

# Hide unused subplots
for i in range(num_snapshots, len(axes)):
    axes[i].axis("off")

plt.tight_layout()
plt.show()

## 12. Export Results

In [ ]:
# Export results to Zarr
output_file = "./SeapoPym_Synthetic.zarr"

biomass.rename("biomass").to_zarr(output_file, mode="w")

print(f"✅ Results exported to {output_file}")